# Perform prediction inference

In [165]:
import numpy as np 
import pandas as pd
import json
import src.network
import src.layer
from src.scaler import StandardScaler
import types 
from  src.scaler import encode_labels 

In [166]:
# def load_model(filepath: str) -> dict:
  # """
  # Safetily load a neural network's topology and waights from a JSON file.
  #   Parameters:
  #       filepath (str): Path to the JSON file containing the model's configuration
  #   Returns:
  #       dict: The parsed model data or empty dict if loading fails
  # """
filepath = '../models/mlp_model.json'
try:

    # utf-8 encoding ensures we can read files with special characters without issues.
    with open(filepath, "r", encoding="utf-8") as f:
        model_data = json.load(f)
    # return model_data

except FileNotFoundError as e:

    print(f"File not found : {e}")
    # return {}

except json.JSONDecodeError as e:

    print(f"Error decoding JSON : {e}")
    # return {}

except Exception as e:

    print(f"An unexpected error occurred : {e}")
    # return {}

In [167]:
model_data

{'topology': [30, 24, 24, 1],
 'layers': [{'activation': 'relu',
   'weights': [[0.18206032965358288,
     -0.04156634136480688,
     0.07245060069786075,
     0.11681813817351665,
     -0.030389591680447702,
     -0.026756710864251396,
     0.19255983142890995,
     0.05040839981235217,
     -0.06311442769793958,
     0.0237763566083049,
     -0.03207574156240588,
     0.070107089321919,
     0.027136931750631553,
     -0.2585347625064886,
     -0.18533394016476018,
     -0.08690514149583914,
     -0.16698922625816026,
     0.13588727598283792,
     -0.1032468172239484,
     -0.1400845944757011,
     0.08118143375362998,
     -0.059750556762316206,
     -0.07470164906684315,
     -0.09497645522337712],
    [0.09060185602611387,
     -0.015889145708928937,
     -0.10886533274662838,
     0.003957959742290598,
     -0.058010058791455696,
     -0.03284764051473007,
     -0.022011605096631554,
     0.16651870317931441,
     -0.014799235696380076,
     -0.1248838877894828,
     0.114862159

In [168]:
model_data['topology']

[30, 24, 24, 1]

In [169]:
for layer in model_data['layers']:
    print(
        f" Activation: {layer['activation']}, Weights shape: {np.array(layer['weights']).shape}, Biases shape: {np.array(layer['biases']).shape}"
    )

 Activation: relu, Weights shape: (30, 24), Biases shape: (1, 24)
 Activation: relu, Weights shape: (24, 24), Biases shape: (1, 24)
 Activation: sigmoid, Weights shape: (24, 1), Biases shape: (1, 1)


# Load scaler

In [170]:
# def load_scaler(filepath: str) -> tuple[np.ndarray, np.ndarray]:
# """
# Loads the starndardization parameters (mean and std dev) from a json file.

# Why we load these paremeters instead of recalculating them : 
# # ──────────────────────────────────────────────────────────
# To prevent data laeakage. In Phase 4 (Prediction/Validation), we must
# transform the new data using the exact same Mean (mu) and Standard Deviation
# (sigma) that were calculated from the training data in Phase 3. This ensures
# that the model's predictions are based on the same feature scaling it was trained on

# Parameters:
#     filepath (str): Path to the JSON file containing the scaler parameters.
# Returns:
#     tuple[np.ndarray, np.ndarray]: A tuple containing the mean and standard deviation arrays.
# """
filepath = 'models/scaler.json'
try:
    with open(filepath, "r", encoding="utf-8") as f:
        scaler_data = json.load(f)
    
    mean = np.array(scaler_data["mean"])
    std = np.array(scaler_data["std"])
    # return mean, std

except FileNotFoundError as e:
    print(f"File not found. Ensure training was completed: {e}")

except KeyError as e:
    print(f"Missing key in scaler JSON : {e}")
    raise SystemExit

except json.JSONDecodeError as e:
    print(f"Error decoding JSON. Ensure the file is valid: {e}")
    raise SystemExit



File not found. Ensure training was completed: [Errno 2] No such file or directory: 'models/scaler.json'


In [171]:
mean, std

(array([1.41321360e+01, 1.92112061e+01, 9.19783333e+01, 6.56054386e+02,
        9.57417105e-02, 1.03642741e-01, 8.90029737e-02, 4.83635987e-02,
        1.81089474e-01, 6.27435746e-02, 4.03704386e-01, 1.20359539e+00,
        2.86897873e+00, 4.04213618e+01, 6.98372807e-03, 2.56116425e-02,
        3.27986862e-02, 1.18882083e-02, 2.05583728e-02, 3.81711908e-03,
        1.62707500e+01, 2.55831140e+01, 1.07337544e+02, 8.82590132e+02,
        1.31550921e-01, 2.52767368e-01, 2.74747213e-01, 1.14295638e-01,
        2.90494079e-01, 8.38641447e-02]),
 array([3.54158588e+00, 4.29309519e+00, 2.43550344e+01, 3.55965464e+02,
        1.38925276e-02, 5.23576876e-02, 7.93249436e-02, 3.80186174e-02,
        2.74276652e-02, 7.19938766e-03, 2.84825806e-01, 5.40905300e-01,
        2.07928591e+00, 4.77204471e+01, 3.05225522e-03, 1.85728491e-02,
        3.20794452e-02, 6.28147979e-03, 8.16040308e-03, 2.78192489e-03,
        4.86054801e+00, 6.13574864e+00, 3.36747228e+01, 5.78898662e+02,
        2.30353168e-02

# Scale val data

## load validation data

In [172]:
try:
    val_raw_data = pd.read_csv('../data/validation_data.csv', header=None)
except FileNotFoundError as e:
    print(f"Validation data file not found: {e}")
    raise SystemExit

In [173]:
val_raw_data

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,87930,B,12.47,18.60,81.09,481.9,0.09965,0.10580,0.08005,0.03821,...,14.97,24.64,96.05,677.9,0.1426,0.2378,0.2671,0.10150,0.3014,0.08750
1,859575,M,18.94,21.31,123.60,1130.0,0.09009,0.10290,0.10800,0.07951,...,24.86,26.58,165.90,1866.0,0.1193,0.2336,0.2687,0.17890,0.2551,0.06589
2,8670,M,15.46,19.48,101.70,748.9,0.10920,0.12230,0.14660,0.08087,...,19.26,26.00,124.90,1156.0,0.1546,0.2394,0.3791,0.15140,0.2837,0.08019
3,907915,B,12.40,17.68,81.47,467.8,0.10540,0.13160,0.07741,0.02799,...,12.88,22.91,89.61,515.8,0.1450,0.2629,0.2403,0.07370,0.2556,0.09359
4,921385,B,11.54,14.44,74.65,402.9,0.09984,0.11200,0.06737,0.02594,...,12.26,19.68,78.78,457.8,0.1345,0.2118,0.1797,0.06918,0.2329,0.08134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,86973702,B,14.44,15.18,93.97,640.1,0.09970,0.10210,0.08487,0.05532,...,15.85,19.85,108.60,766.9,0.1316,0.2735,0.3103,0.15990,0.2691,0.07683
109,913102,B,14.64,16.85,94.21,666.0,0.08641,0.06698,0.05192,0.02791,...,16.46,25.44,106.00,831.0,0.1142,0.2070,0.2437,0.07828,0.2455,0.06596
110,8610404,M,16.07,19.65,104.10,817.7,0.09168,0.08424,0.09769,0.06638,...,19.77,24.56,128.80,1223.0,0.1500,0.2045,0.2829,0.15200,0.2650,0.06387
111,884689,B,11.52,14.93,73.87,406.3,0.10130,0.07808,0.04328,0.02929,...,12.65,21.19,80.88,491.8,0.1389,0.1582,0.1804,0.09608,0.2664,0.07809


In [174]:
# slice the data in to features(x) and targer-lables(y)
# col 0 is the id, col 1 is the targer labe, and the rest 2-31 are the features
# Take all the raws, and columns from index 2 to the end (features)
x_val = val_raw_data.iloc[:, 2:].values

# take all the raws, and column at index 1 (target labels)
y_val = val_raw_data.iloc[:, 1].values

print("x_val shape:", x_val.shape)
print("y_val shape:", y_val.shape)

x_val shape: (113, 30)
y_val shape: (113,)


In [175]:
x_val[1]

array([1.894e+01, 2.131e+01, 1.236e+02, 1.130e+03, 9.009e-02, 1.029e-01,
       1.080e-01, 7.951e-02, 1.582e-01, 5.461e-02, 7.888e-01, 7.975e-01,
       5.486e+00, 9.605e+01, 4.444e-03, 1.652e-02, 2.269e-02, 1.370e-02,
       1.386e-02, 1.698e-03, 2.486e+01, 2.658e+01, 1.659e+02, 1.866e+03,
       1.193e-01, 2.336e-01, 2.687e-01, 1.789e-01, 2.551e-01, 6.589e-02])

In [176]:
y_val[:10]

<StringArray>
['B', 'M', 'M', 'B', 'B', 'M', 'M', 'M', 'B', 'B']
Length: 10, dtype: str

In [177]:
y_val_encoded = encode_labels(y_val).reshape(-1, 1)

In [178]:
y_val_encoded[:10]


array([[0],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [0]])

In [179]:
# we initialize the object scaler 
scaler = StandardScaler()

In [180]:
# We  instantiate the scaler with the loaded mean and std dev
scaler.mean_ = mean
scaler.std_ = std

In [181]:
# we standardize the validation features using the transfrom method of the scaler object
# which is secured against division by zero using the epsilon value
x_val_scaled = scaler.transform(x_val)

In [182]:
x_val[1]

array([1.894e+01, 2.131e+01, 1.236e+02, 1.130e+03, 9.009e-02, 1.029e-01,
       1.080e-01, 7.951e-02, 1.582e-01, 5.461e-02, 7.888e-01, 7.975e-01,
       5.486e+00, 9.605e+01, 4.444e-03, 1.652e-02, 2.269e-02, 1.370e-02,
       1.386e-02, 1.698e-03, 2.486e+01, 2.658e+01, 1.659e+02, 1.866e+03,
       1.193e-01, 2.336e-01, 2.687e-01, 1.789e-01, 2.551e-01, 6.589e-02])

In [183]:
x_val_scaled[1] 

array([ 1.35754551,  0.48887662,  1.29836264,  1.33143707, -0.40681628,
       -0.0141859 ,  0.23948361,  0.81924055, -0.83453933, -1.12975765,
        1.35203901, -0.75076984,  1.2586154 ,  1.16571913, -0.83207975,
       -0.48951228, -0.31511402,  0.28843343, -0.82083747, -0.76174287,
        1.76713613,  0.16247177,  1.73906275,  1.69875996, -0.53183188,
       -0.12392032, -0.02893905,  0.99044582, -0.56169632, -1.00927867])

# Load the model with trainned weights and biases to perform inference (predictions)

In [184]:
from src.network import MultilayerPerceptron

# load Network class and initialize it with random weights and biases based on the loaded topology
model = MultilayerPerceptron(
    topology=model_data["topology"],
    hidden_activation="relu",
    output_activation="sigmoid",
)

In [185]:
model.summary()

Layer (Type)         Shape (In, Out)      Param #        
Dense-1 (relu)       (30, 24)             744            
------------------------------------------------------------
Dense-2 (relu)       (24, 24)             600            
------------------------------------------------------------
Dense-3 (sigmoid)    (24, 1)              25             
------------------------------------------------------------
Total params: 1369


In [186]:
for layer, layer_data in zip(model.layers, model_data["layers"]):
    layer.weights = np.array(layer_data["weights"])
    layer.biases = np.array(layer_data["biases"])

In [187]:
# object we have just created in memory, which is an instance of the MultilayerPerceptron class, and we can access its first layer's weights using the following code:
model.layers[0].weights

array([[ 0.18206033, -0.04156634,  0.0724506 ,  0.11681814, -0.03038959,
        -0.02675671,  0.19255983,  0.0504084 , -0.06311443,  0.02377636,
        -0.03207574,  0.07010709,  0.02713693, -0.25853476, -0.18533394,
        -0.08690514, -0.16698923,  0.13588728, -0.10324682, -0.14008459,
         0.08118143, -0.05975056, -0.07470165, -0.09497646],
       [ 0.09060186, -0.01588915, -0.10886533,  0.00395796, -0.05801006,
        -0.03284764, -0.02201161,  0.1665187 , -0.01479924, -0.12488389,
         0.11486216, -0.01337399,  0.02022256, -0.26330412, -0.13799002,
         0.00456452,  0.02057297,  0.10105222, -0.0212343 , -0.02586499,
        -0.19736874, -0.11098188, -0.09276978,  0.12562321],
       [ 0.16354086, -0.20307253,  0.04030233, -0.07339218, -0.07445513,
         0.0575276 ,  0.13682666,  0.06687507, -0.10068669, -0.06134933,
         0.04644307,  0.21241388, -0.04546806, -0.08587034, -0.12331554,
        -0.15024324,  0.01536924,  0.23841605, -0.02072725,  0.10161651,
  

In [188]:
# Dictionnary with the model's topology and weights for each layer, which can be used to re-initialize the model with the same parameters.
model_data["layers"][0]["weights"]

[[0.18206032965358288,
  -0.04156634136480688,
  0.07245060069786075,
  0.11681813817351665,
  -0.030389591680447702,
  -0.026756710864251396,
  0.19255983142890995,
  0.05040839981235217,
  -0.06311442769793958,
  0.0237763566083049,
  -0.03207574156240588,
  0.070107089321919,
  0.027136931750631553,
  -0.2585347625064886,
  -0.18533394016476018,
  -0.08690514149583914,
  -0.16698922625816026,
  0.13588727598283792,
  -0.1032468172239484,
  -0.1400845944757011,
  0.08118143375362998,
  -0.059750556762316206,
  -0.07470164906684315,
  -0.09497645522337712],
 [0.09060185602611387,
  -0.015889145708928937,
  -0.10886533274662838,
  0.003957959742290598,
  -0.058010058791455696,
  -0.03284764051473007,
  -0.022011605096631554,
  0.16651870317931441,
  -0.014799235696380076,
  -0.1248838877894828,
  0.11486215974310501,
  -0.01337398829131003,
  0.020222556470234104,
  -0.2633041171845536,
  -0.13799002234633545,
  0.00456451541862295,
  0.02057297139805459,
  0.1010522195211072,
  -0.021

In [189]:
y_pred_val = model.forward(x_val_scaled)

In [190]:
from src.loss import binary_cross_entropy


val_loss = binary_cross_entropy(y_val_encoded, y_pred_val)

In [191]:
y_pred_val[:5]

array([[0.11979265],
       [0.98596439],
       [0.94036427],
       [0.05324536],
       [0.01885021]])

In [192]:
val_loss

0.09214463314229729

In [193]:
y_val[:5]

<StringArray>
['B', 'M', 'M', 'B', 'B']
Length: 5, dtype: str

In [194]:
y_val_encoded[:5]

array([[0],
       [1],
       [1],
       [0],
       [0]])

In [195]:
y_pred_val[:5]

array([[0.11979265],
       [0.98596439],
       [0.94036427],
       [0.05324536],
       [0.01885021]])

In [196]:
y_val_encoded[:5]

array([[0],
       [1],
       [1],
       [0],
       [0]])

In [197]:
# (Turns predictions > 0.5 into true otherwise false) then compare with the actual results 
(y_pred_val[:5] > 0.5) == y_val_encoded[:5]

array([[ True],
       [ True],
       [ True],
       [ True],
       [ True]])

In [198]:
(y_pred_val[:5] > 0.5)

array([[False],
       [ True],
       [ True],
       [False],
       [False]])

In [199]:
val_acc = np.mean((y_pred_val > 0.5) == y_val_encoded)

In [200]:
val_acc

np.float64(0.9734513274336283)